In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Install Packages:

In [2]:
!pip install llama-index llama-parse llama-index-llms-groq llama-index-llms-anthropic pydantic pandas chromadb
!pip install llama-index-embeddings-huggingface
!pip install torch

# Module Import

In [3]:
import pandas as pd
import numpy as np
from  matplotlib import pylab as plt

import torch
from datasets import Dataset

import os

## GPU check:

In [4]:
# This should return True if your NVIDIA GPU is recognized
print(torch.cuda.is_available())
# Optional: Check the name of your GPU
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

False
Using device: cpu


# 1. Data Ingestion

In [ ]:
import os
from llama_parse import LlamaParse

os.environ["LLAMA_CLOUD_API_KEY"] = ""

# Cấu hình LlamaParse ưu tiên xử lý bảng biểu phức tạp
parser = LlamaParse(
    result_type="markdown",  # Markdown giúp LLM hiểu cấu trúc bảng tốt hơn
    verbose=True
)

# Đọc báo cáo tài chính (Ví dụ: VNM_Annual_Report.pdf)
documents = parser.load_data("/content/drive/MyDrive/Colab Notebooks/NLP & LLMs/Fundamental Anlysis by Agent/20260429_VNM_BCTC_DA_SOAT_XET_Q1_2026_RIENG_VN_920896fa41.pdf")

Started parsing the file under job_id 1a3c2936-9441-460b-b76e-0fcb429ffc4b


In [6]:
markdown_content = "\n\n---\n\n".join([doc.text for doc in documents])

# 2. Định nghĩa đường dẫn
file_path = "/content/drive/MyDrive/Colab Notebooks/NLP & LLMs/Fundamental Anlysis by Agent/20260429_VNM_BCTC_DA_SOAT_XET_Q1_2026_RIENG_VN_920896fa41.markdown"

# 3. Ghi dữ liệu ra file (Lưu ý: Bắt buộc dùng encoding="utf-8" để không bị lỗi font tiếng Việt)
with open(file_path, "w", encoding="utf-8") as file:
    file.write(markdown_content)

print(f"Đã xuất thành công dữ liệu BCTC ra file: {file_path}")

Đã xuất thành công dữ liệu BCTC ra file: /content/drive/MyDrive/Colab Notebooks/NLP & LLMs/Fundamental Anlysis by Agent/20260429_VNM_BCTC_DA_SOAT_XET_Q1_2026_RIENG_VN_920896fa41.markdown


# 2. Vector Search

In [ ]:
import os
from llama_index.llms.groq import Groq
from llama_index.core import VectorStoreIndex, Settings
from llama_index.llms.openai import OpenAI
from llama_index.core.node_parser import SentenceSplitter
# Import thêm module nhúng mã nguồn mở miễn phí
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# 1. Cấu hình Key và trỏ Base URL về máy chủ miễn phí của Groq
os.environ["OPENAI_API_KEY"] = ""

'''
Settings.llm = OpenAI(
    model="llama-3.3-70b-versatile",
    #api_base="https://api.groq.com/openai/v1",
    temperature=0
)
'''
Settings.llm = Groq(
    model="llama-3.3-70b-versatile", # Đổi sang mô hình này, nó có hạn mức cao hơn
    temperature=0
)

# 2. BẮT BUỘC: Khai báo mô hình nhúng miễn phí (Chạy local trên máy/Colab)
#Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
#Settings.embed_model = HuggingFaceEmbedding(model_name="keepitreal/vietnamese-sbert")
Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

Settings.text_splitter = SentenceSplitter(chunk_size=1024, chunk_overlap=200)

# 3. Bây giờ bạn có thể quét tài liệu và tạo Index mà không tốn một xu
index = VectorStoreIndex.from_documents(documents)

# 3. Structured Output

In [32]:
import os
from pydantic import BaseModel, Field
from typing import Optional

# 1. Định nghĩa cấu trúc dữ liệu đầu ra (Schema) - Giữ nguyên cấu trúc của bạn
class ComprehensiveFinancialMetrics(BaseModel):
    # Thông tin cơ bản
    Fiscal_Year: int = Field(description="Năm tài chính của số liệu được trích xuất (ví dụ: 2023).")
    Ticker: str = Field(description="Mã cổ phiếu của công ty (ví dụ: VNM, DIG).")

    # Báo cáo kết quả kinh doanh
    Revenue: Optional[int] = Field(description="Doanh thu thuần về bán hàng và cung cấp dịch vụ.")
    Gross_Profit: Optional[int] = Field(description="Lợi nhuận gộp về bán hàng và cung cấp dịch vụ.")
    EBIT: Optional[int] = Field(description="Lợi nhuận thuần từ hoạt động kinh doanh trước lãi vay và thuế.")
    Net_Income: Optional[int] = Field(description="Lợi nhuận sau thuế thu nhập doanh nghiệp.")

    # Bảng cân đối kế toán
    Cash_And_Equivalents: Optional[int] = Field(description="Tiền và các khoản tương đương tiền tại thời điểm cuối năm.")
    Short_Term_Debt: Optional[int] = Field(description="Vay và nợ thuê tài chính ngắn hạn.")
    Long_Term_Debt: Optional[int] = Field(description="Vay và nợ thuê tài chính dài hạn.")
    Total_Equity: Optional[int] = Field(description="Tổng vốn chủ sở hữu.")

    # Báo cáo lưu chuyển tiền tệ & Biến động
    CFO: Optional[int] = Field(description="Lưu chuyển tiền thuần từ hoạt động kinh doanh.")
    CapEx: Optional[int] = Field(description="Tiền chi để mua sắm, xây dựng TSCĐ và các tài sản dài hạn khác (thường mang giá trị âm trong báo cáo, hãy chuyển thành số nguyên dương).")
    Depreciation: Optional[int] = Field(description="Chi phí khấu hao tài sản cố định trong năm.")

    # Dữ liệu thị trường
    Shares_Outstanding: Optional[int] = Field(description="Tổng số lượng cổ phiếu đang lưu hành tại thời điểm báo cáo.")

# 2. Khởi tạo bộ định dạng mẫu của LlamaIndex
output_parser = PydanticOutputParser(ComprehensiveFinancialMetrics)

# --- THÊM DÒNG NÀY ĐỂ KHỞI TẠO QUERY ENGINE TỪ INDEX BẠN ĐÃ TẠO ---
query_engine = index.as_query_engine(similarity_top_k=7)

# 4. Integration

In [34]:
import os
import re
import json
import pandas as pd
from pydantic import BaseModel, Field
from typing import Optional
from llama_index.core.output_parsers import PydanticOutputParser

# 3. ĐỒNG BỘ HÓA JSON TEMPLATE: Cập nhật đầy đủ 14 trường cho khớp với Pydantic Class
json_template = {
    "Fiscal_Year": 2023,
    "Ticker": "VNM",
    "Revenue": 1000000000,
    "Gross_Profit": 500000000,
    "EBIT": 200000000,
    "Net_Income": 1844322488803,
    "Cash_And_Equivalents": 100000000,
    "Short_Term_Debt": 50000000,
    "Long_Term_Debt": 20000000,
    "Total_Equity": 300000000,
    "CFO": 450000000,
    "CapEx": 120000000,
    "Depreciation": 10000000,
    "Shares_Outstanding": 1500000
}

# 4. CẬP NHẬT CÂU LỆNH (PROMPT): Mở rộng phạm vi tìm kiếm cho LLM
query_str = f"""
Hãy quét toàn bộ Báo cáo tài chính và trích xuất các số liệu tài chính tương ứng.
Dưới đây là một số hướng dẫn để tìm kiếm:
- Báo cáo kết quả kinh doanh: Tìm Doanh thu (Revenue), Lợi nhuận gộp (Gross_Profit), EBIT, Lợi nhuận sau thuế (Net_Income).
- Bảng cân đối kế toán: Tìm Tiền & tương đương tiền, Nợ ngắn hạn, Nợ dài hạn, Tổng vốn chủ sở hữu.
- Báo cáo lưu chuyển tiền tệ: Tìm Lưu chuyển tiền thuần HĐKD (CFO), Tiền chi mua sắm TSCĐ (CapEx), Khấu hao.
- Tự động nhận diện Năm tài chính (Fiscal_Year) và Mã cổ phiếu (Ticker) nếu có. Nếu không thấy, hãy trả về null đối với số, hoặc "Unknown" đối với chuỗi.

YÊU CẦU BẮT BUỘC VỀ ĐỊNH DẠNG ĐẦU RA:
- Trả về DUY NHẤT một chuỗi JSON hợp lệ.
- JSON phải chứa ĐẦY ĐỦ tất cả các Key có trong mẫu dưới đây (nếu không có số liệu, điền null).
- KHÔNG viết thêm bất kỳ văn bản dẫn giải nào bên ngoài khối JSON.

Cấu trúc định dạng JSON bắt buộc:
{json.dumps(json_template, indent=2, ensure_ascii=False)}
"""

# 5. Thực hiện truy vấn qua Query Engine
print("Mô hình đang thực thi cấu trúc truy vấn nâng cao với 14 chỉ số...")
response = query_engine.query(query_str)
raw_text = response.response

# 6. Hàm bóc tách JSON
def extract_and_validate_json(text, pydantic_cls):
    json_match = re.search(r"\{.*\}", text, re.DOTALL)
    if not json_match:
        raise ValueError("Hệ thống không tìm thấy cấu trúc JSON hợp lệ.")
    return pydantic_cls.model_validate_json(json_match.group())

# 7. Biên dịch kết quả về cấu trúc DataFrame động (Tự động lấy tất cả các trường)
try:
    financial_data = extract_and_validate_json(raw_text, ComprehensiveFinancialMetrics)

    # Chuyển đổi Pydantic object thành Dictionary để đưa vào DataFrame
    data_dict = financial_data.model_dump()

    df = pd.DataFrame({
        "Chỉ số": list(data_dict.keys()),
        "Giá trị": list(data_dict.values())
    })

    # Format số cho đẹp (bỏ qua các trường string như Ticker)
    def format_value(x):
        if isinstance(x, (int, float)) and pd.notna(x):
            return f"{x:,.0f}"
        elif x is None:
            return "Trống"
        return str(x)

    df["Giá trị"] = df["Giá trị"].apply(format_value)

    print("\n--- KẾT QUẢ TRÍCH XUẤT THÀNH CÔNG ---")
    print(df.to_string(index=False))

except Exception as e:
    print(f"\nQuá trình phân tách gặp lỗi định dạng: {e}")
    print("Nội dung văn bản thô nhận được từ LLM:\n", raw_text)

Mô hình đang thực thi cấu trúc truy vấn nâng cao với 14 chỉ số...

--- KẾT QUẢ TRÍCH XUẤT THÀNH CÔNG ---
              Chỉ số         Giá trị
         Fiscal_Year           2,026
              Ticker         Unknown
             Revenue           Trống
        Gross_Profit           Trống
                EBIT           Trống
          Net_Income           Trống
Cash_And_Equivalents           Trống
     Short_Term_Debt           Trống
      Long_Term_Debt 605,923,010,181
        Total_Equity           Trống
                 CFO           Trống
               CapEx     979,670,867
        Depreciation           Trống
  Shares_Outstanding           Trống
